### 1. Definicion del problema de clasificacion
- Problema:
R: Se necesita predecir, dado el repositorio publico, si te aprobaran un pull request hacia el repositorio.
Se plantea en base a la necesidad de conocer la configuracion optima junto a los estandares especificados para conocer si un pull request sera efectivamente recibido de forma positiva por el equipo de desarrollo, para esto se ha recopilado un dataset proveniente de la API publica proporcionada por GitHub.

Se ha establecido un problema de clasificacion binario, en donde predeciremos si dada cierta cantidad de informacion correspondiente a un pull request este sera aceptado o no.

Data la complejidad, naturaleza y variedad en los repositorios, los pull request pueden variar en su formato, se vuelve entonces un problema predecir de forma directa si un pull request sera aceptado solo con observarlo, asi entonces con el análisis de multiples pull request en torno a repositorios relacionados se espera obtener una modelo capaz de predecir el destino de un pull request.

Se ha hecho uso de la API publica de github y se ha extraído 30000 datos correspondientes a pull request sobre 8 repositorios relacionados al entorno FrontEnd en el desarrollo de software.
En primera instancia se extrajo una gran cantidad de columnas (40+) con el objetivo de tener la informacion suficiente para el entrenamiento, luego se han renombrado y se ha hecho un segundo tipo de filtrado para obtener valores relevantes. Sin embargo la gran mayoria de los datos corresponde a valores de tipo string, se ha hecho una transformacion y filtrado adicional, transformando la totalidad de los datos a valores numericos priorizando la creacion de valores binarios para facilitar el trabajo del clasificador.

Luego, se da paso a la etapa de entrenamiento, comenzamos comparando la distribucion de nuestro dataset y se realiza un balance, al mismo tiempo, se entrenan multiples modelos utilizando una variedad de hiperparametros con el objetivo de encontrar el modelo y distribucion que favorece y entrega un resultado que se acomoda al problema planteado.



### 2. Diseño del experimento:


#### Extracción de datos

#### Explicacion
R: Los datos se dividieron de la siguiente forma:
El entrenamiento y prueba lo que se hizo fue tomar el dataset generado, el csv y particionarlo en variables de entrenamiento y prueba:
Cargar los datos:
```py
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
```

Generar datos de prueba y validacion:
```py

X = data.iloc[:,:-1].values
y = data['merged'].values

X_train, X_test, y_train, y_test = train_test_split(
   X, 
   y, 
   test_size=.30,
   random_state=15, 
   stratify=y
)
```

In [ ]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
data.head(1)

### checks initial initial class distribution
data['merged'].value_counts()

In [11]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

In [ ]:
load_dotenv()
github_key = os.getenv("GITHUB_TOKEN")

headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {github_key}"
}

urls: list[str] = []

urls.append("https://api.github.com/repos/nodejs/node/pulls?state=all")
urls.append("https://api.github.com/repos/angular/angular/pulls?state=all")
urls.append("https://api.github.com/repos/vuejs/core/pulls?state=all")
urls.append("https://api.github.com/repos/vercel/next.js/pulls?state=all")
urls.append("https://api.github.com/repos/facebook/react/pulls?state=all")
urls.append("https://api.github.com/repos/sveltejs/svelte/pulls?state=all")
urls.append("https://api.github.com/repos/withastro/astro/pulls?state=all")
urls.append("https://api.github.com/repos/QwikDev/qwik/pulls?state=all")

Se extraen datos con la utilización de API Github
El análisis se ha realizado utilizando 125 paginas, se recomienda bajar este numero para realizar la prueba.

In [ ]:
### numero de paginas a buscar, se recomienda baja este numero
max_pages = 125

def get_data(urls: list[str]) -> list[str]:
    all_results: list[str] = []
    for url in urls:
        page = 0
        print(f'Current url: {url}')
        while url:
            page += 1
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                print(f'Current page: {page}')
                data = response.json()
                
                all_results.extend(data)

                link_header = response.headers.get("Link", "")
                next_url = None
                for link in link_header.split(","):
                    if 'rel="next"' in link:
                        next_url = link[link.find("<")+1:link.find(">")]
                        break

                url = next_url

                # early return
                if page == max_pages:
                    url = None

                # 'don't get banned' check
                time.sleep(0.3)
                
            elif response.status_code == 202:
                print("Compiling data, try again shortly")
                break
            else:
                print(f"Error: {response.status_code}")
                break
    return all_results

In [ ]:
all_results = get_data(urls)

Se transforma la informacion obtenida

In [ ]:
df = pd.json_normalize(
    all_results, 
    record_path=None, 
    meta=None, 
    errors='ignore'
)
df.tail(10)

Se limpian columnas que en su totalidad sean null para evitar valores indeseados y se filtran elementos deseados.

In [ ]:
cleaned_df = df.dropna(axis=1, how='all')
filtered_df = cleaned_df[
   [
      "number", 
      "state", 
      "title", 
      "body", 
      "locked",
      "created_at",
      "updated_at",
      "closed_at",
      "merged_at",
      "assignees",
      "user.login",
      "labels",
      "author_association",
      "user.repos_url",
      "user.followers_url",
      "user.organizations_url",
      "user.starred_url",
      "user.type",
      "base.user.login",
      'base.repo.name',
      "base.user.followers_url",
      "base.user.starred_url",
      "base.repo.created_at",
      "base.repo.updated_at", 
      "base.repo.pushed_at",
      "base.repo.size",
      "base.repo.releases_url",
      "base.repo.stargazers_count",
      "base.repo.watchers_count",
      "base.repo.language",
      "base.repo.has_issues", 
      "base.repo.has_projects",
      "base.repo.has_downloads",
      "base.repo.has_wiki",
      "base.repo.has_pages",
      "base.repo.has_discussions",
      "base.repo.forks_count"
   ]
]

Se realiza un filtrado sobre las columnas sin url, en un comienzo se tenia planteado obtener aun mas información, pero la cantidad de peticiones necesarias incrementaba demasiado, se opta por filtrar.

In [ ]:
non_url_columns_df = filtered_df.drop(columns=[col for col in filtered_df.columns if col.endswith('_url')])

Se renombra columnas para facilitar su manejo.

In [ ]:
non_url_columns_df.columns = [
   'number',
   'state',
   'title',
   'body',
   'locked',
   'created_at',
   'updated_at',
   'closed_at',
   'merged_at',
   'assignees',
   'user_name',
   'labels',
   'author_association',
   'user_type',
   'repo_owner_name',
   'repo_name',
   'repo_created_at',
   'repo_updated_at',
   'repo_pushed_at',
   'repo_size',
   'repo_stargazer_count',
   'repo_watcher_count',
   'repo_language',
   'repo_has_issue',
   'repo_has_projects',
   'repo_has_downloads',
   'repo_has_wiki',
   'repo_has_pages',
   'repo_has_discussions',
   'repo_fork_count'
]

Se guarda la información obtenida para su futura manipulación

In [ ]:
non_url_columns_df.to_csv("../data/dataset/pull_request_data.csv", index=False)

## MANEJO/PREPARACIÓN DE DATOS

In [2]:
import pandas as pd
import numpy as np
import ast

Se carga dataset previamente guardado.

In [3]:
csv_path = '../data/dataset/pull_request_data.csv'
data = pd.read_csv(csv_path, index_col=False)

Observamos composición de columnas para entender con que tipo de columnas se trabajara

In [4]:
data.columns

Index(['number', 'state', 'title', 'body', 'locked', 'created_at',
       'updated_at', 'closed_at', 'merged_at', 'assignees', 'user_name',
       'labels', 'author_association', 'user_type', 'repo_owner_name',
       'repo_name', 'repo_created_at', 'repo_updated_at', 'repo_pushed_at',
       'repo_size', 'repo_stargazer_count', 'repo_watcher_count',
       'repo_language', 'repo_has_issue', 'repo_has_projects',
       'repo_has_downloads', 'repo_has_wiki', 'repo_has_pages',
       'repo_has_discussions', 'repo_fork_count'],
      dtype='object')

Se analiza tamaño de dataset, para considerar 

In [5]:
data.shape

(30000, 30)

Se crean clases iniciales, se considera:
- Merged : 1
- Not Merged : 0

In [6]:
### creates clases
## 1: MERGED
## 0: NOT MERGED
merged_at = data['merged_at']
merged = merged_at.notna().map({True: 1, False: 0})
data.insert(len(data.columns), 'merged', merged)

Para permitir un entrenamiento se ha hecho una modificacion de las columnas del dataset, se ha modicado de forma en que la gran mayoria de los datos se vuelve binario y la totalidad numerico.

In [7]:
### manages state
data["state"] = data["state"].map({'open': 1, 'closed': 0})
data.rename(columns={"state": "state_open"}, inplace=True)

### manages title
data["title"] = data["title"].notna().map({True: 1, False: 0})
data.rename(columns={"title": "has_title"}, inplace=True)

### manages body
data["body"] = data["body"].notna().map({True: 1, False: 0})
data.rename(columns={"body": "has_body"}, inplace=True)

### manages locked
data["locked"] = data["locked"].map({True: 1, False: 0})
data.rename(columns={"locked": "is_locked"}, inplace=True)

### manages closed_at
data["closed_at"] = data["closed_at"].notna().map({True: 1, False: 0})
data.rename(columns={"closed_at": "is_closed"}, inplace=True)

### manages assignees
data["assignees"] = data["assignees"].apply(lambda x: bool(ast.literal_eval(x))).map({True: 1, False: 0})
data.rename(columns={"assignees": "has_assignees"}, inplace=True)

### manages labels
data["labels"] = data["labels"].apply(lambda x: bool(ast.literal_eval(x))).map({True: 1, False: 0})
data.rename(columns={"labels": "has_labels"}, inplace=True)

### manages author
data["author_association"] = data["author_association"].map({
    'NONE': 0,
    'MEMBER': 1,
    'CONTRIBUTOR': 2,
    "COLLABORATOR": 3
})

### manages user_type
data["user_type"] = data["user_type"].map({
    'User': 0,
    'Bot': 1,
})

### manages repo_issue
data["repo_has_issue"] = data["repo_has_issue"].map({True: 1, False: 0})

### manages repo_project
data["repo_has_projects"] = data["repo_has_projects"].map({True: 1, False: 0})

### manages repo_downloads
data["repo_has_downloads"] = data["repo_has_downloads"].map({True: 1, False: 0})

### manages repo_wiki
data["repo_has_wiki"] = data["repo_has_wiki"].map({True: 1, False: 0})

### manages repo_pages
data["repo_has_pages"] = data["repo_has_pages"].map({True: 1, False: 0})

### manages repo_discussions
data["repo_has_discussions"] = data["repo_has_discussions"].map({True: 1, False: 0})

Se elimina columnas no utilizadas debido a su complejidad al momento su transformacion a valor numerico.

In [8]:
data.drop(columns=["number", "created_at", "updated_at", "merged_at", 'user_name', "repo_owner_name", "repo_name", "repo_created_at", "repo_updated_at", "repo_pushed_at", "repo_language"], inplace=True)

Finalmente se guarda el dataset modificado y preparado para entrenamiento.

In [9]:
save_path = '../data/dataset/pull_request_data_processed.csv'
data.to_csv(save_path, index=False)
data

,state_open,has_title,has_body,is_locked,is_closed,has_assignees,has_labels,author_association,user_type,repo_size,repo_stargazer_count,repo_watcher_count,repo_has_issue,repo_has_projects,repo_has_downloads,repo_has_wiki,repo_has_pages,repo_has_discussions,repo_fork_count,merged
0,1,1,1,0,0,0,1,0,0,1335380,111257,111257,1,1,1,0,0,0,31553,0
1,1,1,1,0,0,0,1,1,0,1335380,111257,111257,1,1,1,0,0,0,31553,0
2,1,1,1,0,0,0,1,1,0,1335380,111257,111257,1,1,1,0,0,0,31553,0
3,0,1,1,0,1,0,1,0,0,1335380,111257,111257,1,1,1,0,0,0,31553,0
4,1,1,1,0,0,0,1,2,0,1335380,111257,111257,1,1,1,0,0,0,31553,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,0,1,1,0,1,0,0,2,0,54238,21357,21357,1,1,1,0,0,0,1347,1
29996,0,1,1,0,1,0,0,2,0,54238,21357,21357,1,1,1,0,0,0,1347,1
29997,0,1,1,0,1,0,0,2,0,54238,21357,21357,1,1,1,0,0,0,1347,1
29998,0,1,1,0,1,0,0,2,0,54238,21357,21357,1,1,1,0,0,0,1347,1


### ENTRENAMIENTO:


In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

Se cargan datos de entrenamiento

In [ ]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
data.head(1)


En primera instancia se analiza la distribucion del dataset, dependiendo de esto se considerara realizar tecnicas de undersampling o oversampling

In [ ]:
### checks initial initial class distribution
data['merged'].value_counts()

Se realiza la separacion de datos entre clases y atributos, ademas, se realiza la separacion de datos de entrenamiento y testeo, esto es importante para lograr fidelidad el proceso de validacion del modelo evitando realizar prediciones sobre datos con los que se ha entrenado el modelo.

In [ ]:
X = data.iloc[:,:-1].values
y = data['merged'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30,
                                                    random_state=15, stratify=y)

Se realiza un entrenamiento inicial de prueba.

In [ ]:

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3,5,7,10]
}

scoring = 'f1'

clf = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, scoring=scoring, cv=10)

In [ ]:
clf.fit(X_train, y_train)


print("Mejor combinación de parámetros:")
print(clf.best_params_)
 
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

#### Preparacion modelos

Preparación de multiples modelos, se utilizan multiples hiperparametros para que luego mediante GridSearch se encuentre el modelo que mejor se ajusta a nuestro Dataset

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
### DUMMY (baseline)

param_grid_dm = {
    'strategy': ['most_frequent', 'stratified', 'uniform']
}

scoring = 'f1'

clf_dm = GridSearchCV(DummyClassifier(), param_grid=param_grid_dm, scoring=scoring, cv=10)

In [ ]:
### DECISION TREE

param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

scoring = 'f1'

clf_dt = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid_dt, scoring=scoring, cv=10)

In [ ]:
### RANDOM FOREST

param_grid_rf = {
    'n_estimators': [50],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'criterion': ['gini', 'entropy']
}

clf_rf = GridSearchCV(RandomForestClassifier(), param_grid=param_grid_rf, scoring='f1', cv=10)


In [ ]:
### SUPPORT VECTOR CLASSIFIER

param_grid_svc = {
    'C': [1],          
    'kernel': ['rbf'],
}

clf_svc = GridSearchCV(SVC(), param_grid=param_grid_svc, scoring='f1', cv=10)


In [ ]:
### GAUSSIANNB

param_grid_nb = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
}

clf_nb = GridSearchCV(GaussianNB(), param_grid=param_grid_nb, scoring='f1', cv=10)


In [ ]:
### K-NEIGHBORS

param_grid_knn = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

clf_knn = GridSearchCV(KNeighborsClassifier(), param_grid=param_grid_knn, scoring='f1', cv=10)


In [ ]:
classifiers = {
    "Base dummy": clf_dm,
    "Decision Tree": clf_dt, 
    "Random Forest": clf_rf, 
    "Support Vector Classifier": clf_svc, 
    "GaussianNB": clf_nb, 
    "K-neighbors": clf_knn
}

Se prepara funcion que estara a cargo de el entrenamiento de todos los modelos creados, tomar como parametros los datos de entrenamiento y muestra de forma comprensiva los resultados, esto permitirá que podamos analizar y comparar los resultados obtenidos.

In [ ]:
def train_with_multiples_models(classifiers: dict, X_train, y_train, X_test, y_test):
    for name, clf in classifiers.items():
        print(':::::::::::::::::::::::::::::::::::::::::')
        print(f'Current classifier: {name}')
        print('Training..')
        clf.fit(X_train, y_train)

        print("Mejor combinación de parámetros:")
        print(clf.best_params_)
        
        y_pred = clf.predict(X_test)
        print(classification_report(y_test, y_pred))

#### 2.3 Manejo de clases desbalanceadas:

En este caso, nosotros queriamos generar multiples modelos para visualizar su rendimiento acorde al balaceo de datos.

Por lo que, se escogio utilizar una de las tecnicas de balanceo de datos, para que los modelos puedan responder mejor ante otras clases.

Tecnicas:
- Undersample.
- Oversample.

In [ ]:


csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)
data['merged'].value_counts()
X = data.drop(columns=['merged'])
y = data['merged']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30, random_state=15, stratify=y)

#### UnderSample

Se realiza Undersample y se observa el resultado

In [ ]:
# Undersample.
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print("Balanced class distribution (undersampling):")
print(y_resampled.value_counts())



Se realiza el entrenamiento con multiples modelos.

In [ ]:
train_with_multiples_models(classifiers=classifiers, X_train=X_resampled, y_train=y_resampled, X_test=X_test, y_test=y_test)

#### OverSample

Se realiza Oversample y se observa el resultado

In [ ]:
# Oversample.
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

print("Balanced class distribution (oversampling):")
print(y_resampled.value_counts())


Se realiza el entrenamiento con multiples modelos.

In [ ]:
train_with_multiples_models(classifiers=classifiers, X_train=X_resampled, y_train=y_resampled, X_test=X_test, y_test=y_test)